In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pickle
import warnings

warnings.filterwarnings("ignore")

# Load data
data_path = "s3://future-focus-prerakmahajan/FutureFocus Spreadsheet.csv"
df = pd.read_csv(data_path, header=0, skipinitialspace=True)
df.columns = df.columns.str.strip()

# Columns to encode
columns_to_encode = [
    "Field of Interest", "1st Major Choice", "2nd Major Choice", 
    "1st Hobby Interest", "2nd Hobby Interest", 
    "1st Strong School Subject", "2nd Strong School Subject",
    "1st Strongest Participated EC", "2nd Most Participated EC", "3rd Most Participated EC",
    "1st EC Category", "2nd EC Category", "3rd EC Category"
]

# Encode categorical columns
feature_encoders = {}
for col in columns_to_encode:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    feature_encoders[col] = le

# Define features and target
X = df[[
    "Field of Interest", "1st Major Choice", "2nd Major Choice", 
    "1st Hobby Interest", 
    "1st Strong School Subject", "2nd Strong School Subject"
]]
y = df["1st Strongest Participated EC"]

# Save target encoder
le_target = feature_encoders["1st Strongest Participated EC"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale input features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = DecisionTreeClassifier(criterion="gini")
model.fit(X_train_scaled, y_train)

# Evaluate accuracy
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Save model & encoders
with open("feature_encoders.pkl", "wb") as f:
    pickle.dump(feature_encoders, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("decision_tree_model.pkl", "wb") as f:
    pickle.dump(model, f)

✅ Model Accuracy: 0.83
